# Arabic-Aware Support Ticket Classifier
**SDAIA Academy · Applied NLP Engineering · Capstone** — Bandar

Classifies customer-support messages (English, Arabic MSA, Gulf dialect) into **27 intents**, and routes low-confidence tickets to a human.

**Problem.** The course baseline (e5 embeddings + logistic regression, trained on English only) scores **0.96 macro-F1 on English but only 0.26 on Arabic and 0.04 on Gulf dialect**: on Arabic its confidence collapses and it abstains on most tickets.

**What this notebook does**

| § | Step | Course lab |
|---|---|---|
| 2 | Leak-free splits: Arabic dev (choose τ) vs Arabic test (report only) | D2_4 |
| 3–4 | Synthetic Arabic training tickets + near-duplicate leakage check | D3_1 |
| 5–7 | Train and compare 4 versions (M0–M3) on the **same** test set, with bootstrap CIs | D1_4, D3_4 |
| 8 | Model selection: 4 text encoders, accuracy vs latency | D2_4 |
| 9 | Error analysis: 12 categorized failures | D3_4 |
| 10 | Save results and the final model for the demo | D3_4 |

Run with **Runtime → Run all** (CPU is fine; §8 takes ~10 min).

## 1 · Setup

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/SDAIAAcademy/applied-nlp.git"
if not Path("/content/applied-nlp").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/applied-nlp"], check=True)
os.chdir("/content/applied-nlp/labs")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
if not Path("data/tickets_train.csv").exists():
    subprocess.run([sys.executable, "prepare_data.py"], check=True)
sys.path.insert(0, os.getcwd())

import io, time, json
import numpy as np, pandas as pd
import nlp_utils as U
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
U.set_seed(42)
OUT = Path("results"); OUT.mkdir(exist_ok=True)
print("✅ Setup done")

✅ Setup done


## 2 · Data and leak-free splits
- **English**: course Bitext tickets — train 5,400 · val 1,080 · test 1,080
- **Arabic**: 108 hand-written tickets (4 per intent: 2 MSA + 2 Gulf). We split them:
  - **AR dev (54)** — used **only** to choose the confidence threshold τ
  - **AR test (54)** — used **only** for the final report

Choosing τ on the test tickets would make results look better than they are (data leakage).

In [2]:
train   = U.load_tickets("train")
val     = U.load_tickets("val")
en_test = U.load_tickets("test")
ar_all  = U.load_tickets("ar_test")

ar_all["rank"] = ar_all.groupby(["intent", "dialect"]).cumcount()
ar_dev  = ar_all[ar_all["rank"] == 0].reset_index(drop=True)
ar_test = ar_all[ar_all["rank"] == 1].reset_index(drop=True)

print(f"train {len(train)} | val {len(val)} | EN test {len(en_test)} | AR dev {len(ar_dev)} | AR test {len(ar_test)}")
print("AR test dialects:", ar_test.dialect.value_counts().to_dict())

train 5400 | val 1080 | EN test 1080 | AR dev 54 | AR test 54
AR test dialects: {'gulf': 27, 'msa': 27}


## 3 · Synthetic Arabic training tickets
162 tickets (6 per intent: 3 MSA + 3 Gulf) written with an LLM (Claude) for the same 27 intents — the ✱ Stretch idea from Lab D3_1.

Rules: **training only, never evaluation**; spot-checked by hand; checked for near-duplicates against every Arabic dev/test ticket (§4).

In [3]:
SYNTH_CSV = '''text,intent,dialect
أود إلغاء طلبي الذي قمت به أمس,cancel_order,msa
هل بالإمكان إيقاف الطلب وإلغاؤه قبل التجهيز؟,cancel_order,msa
لا أحتاج الطلب بعد الآن، ألغوه من فضلكم,cancel_order,msa
كنسلوا الطلب الله يعافيكم ما عاد احتاجه,cancel_order,gulf
بغيت الغي الطلبية اللي طلبتها اليوم,cancel_order,gulf
تكفون الغوا طلبي غيرت رايي,cancel_order,gulf
أحتاج إلى تغيير أحد المنتجات في طلبي,change_order,msa
هل يمكن استبدال المنتج في الطلب بلون آخر قبل الشحن؟,change_order,msa
أريد حذف منتج من الطلب الذي لم يُشحن بعد,change_order,gulf
ابي ازيد حبة ثانية على نفس الطلب,change_order,gulf
طلبت مقاس كبير وابيه وسط عدلوه لي,change_order,gulf
أرغب في تعديل محتويات طلبي الحالي,change_order,msa
عنوان الشحن في طلبي خاطئ وأريد تصحيحه,change_shipping_address,msa
هل يمكن توجيه الشحنة إلى مدينة أخرى؟,change_shipping_address,msa
أرجو تحديث عنوان التسليم للطلب قبل خروجه,change_shipping_address,msa
ابي الطلب يروح لبيت اهلي بدل بيتي,change_shipping_address,gulf
حطيت العنوان غلط عدلوه لي الله يخليكم,change_shipping_address,gulf
غيروا عنوان الطلب للرياض بدال جدة,change_shipping_address,gulf
هل سيتم خصم مبلغ إذا ألغيت الطلب الآن؟,check_cancellation_fee,msa
ما هي تكلفة إلغاء الحجز بعد تأكيده؟,check_cancellation_fee,msa
أريد معرفة الرسوم المترتبة على الإلغاء,check_cancellation_fee,msa
لو الغيت الطلب كم بيخصمون مني؟,check_cancellation_fee,gulf
الالغاء ببلاش ولا عليه رسوم؟,check_cancellation_fee,gulf
فيه رسوم لو كنسلت بعد ما يطلع الطلب؟,check_cancellation_fee,gulf
أرغب في الاطلاع على تفاصيل فاتورتي,check_invoice,msa
يوجد خطأ في مبلغ الفاتورة وأريد التحقق منه,check_invoice,msa
لماذا قيمة الفاتورة أعلى من سعر المنتج؟,check_invoice,msa
الفاتورة طالعة غلط ابي اراجعها,check_invoice,gulf
وش هالمبلغ الزايد اللي في الفاتورة؟,check_invoice,gulf
ابي اتأكد من الفاتورة قبل ادفع,check_invoice,gulf
هل تقبلون الدفع بالبطاقة الائتمانية؟,check_payment_methods,msa
أريد معرفة وسائل الدفع المتوفرة,check_payment_methods,msa
هل يمكن الدفع بالتقسيط عن طريق تابي؟,check_payment_methods,msa
وش طرق الدفع اللي عندكم؟,check_payment_methods,gulf
ينفع ادفع كاش لما يوصلني الطلب؟,check_payment_methods,gulf
تقبلون STC Pay؟,check_payment_methods,gulf
ما شروط إرجاع المنتجات لديكم؟,check_refund_policy,msa
هل يمكن إرجاع المنتج بعد فتح العبوة؟,check_refund_policy,msa
كم مدة الاسترجاع المسموح بها؟,check_refund_policy,msa
وش سياستكم في الترجيع؟,check_refund_policy,gulf
اقدر ارجع شي اشتريته من شهر؟,check_refund_policy,gulf
المنتجات اللي عليها خصم ترجع ولا لا؟,check_refund_policy,gulf
أنا غير راضٍ عن مستوى الخدمة إطلاقاً,complaint,msa
أود الإبلاغ عن سوء تعامل من مندوب التوصيل,complaint,msa
تأخر طلبي كثيراً ولم يتواصل معي أحد,complaint,msa
خدمتكم تعبانة مرة وما احد يرد,complaint,gulf
المندوب كان قليل ادب معي ابي اشتكي,complaint,gulf
زهقت منكم كل مرة نفس المشكلة,complaint,gulf
ما هو رقم التواصل مع خدمة العملاء؟,contact_customer_service,msa
أحتاج إلى التواصل مع فريق الدعم,contact_customer_service,msa
هل يوجد بريد إلكتروني لخدمة العملاء؟,contact_customer_service,msa
كيف اكلم الدعم عندكم؟,contact_customer_service,gulf
عطوني ايميل خدمة العملاء,contact_customer_service,gulf
وش رقم الواتساب حقكم؟,contact_customer_service,gulf
أريد محادثة شخص من الدعم وليس الرد الآلي,contact_human_agent,msa
من فضلك حوّلني إلى موظف خدمة العملاء,contact_human_agent,msa
لا أريد التحدث مع البوت، أريد إنساناً,contact_human_agent,msa
ابي موظف يكلمني مو آلة,contact_human_agent,gulf
وصلني بأحد من الدعم الله يعافيك,contact_human_agent,gulf
ما ابي البوت ابي اكلم شخص,contact_human_agent,gulf
كيف يمكنني التسجيل كعميل جديد؟,create_account,msa
أود فتح حساب في متجركم,create_account,msa
ما المطلوب لإنشاء حساب؟,create_account,msa
ابي اسجل عندكم كيف؟,create_account,gulf
ودي اسوي حساب جديد برقم جوالي,create_account,gulf
كيف افتح لي حساب في التطبيق؟,create_account,gulf
أريد إلغاء حسابي وحذف بياناتي,delete_account,msa
كيف يمكنني إزالة حسابي بشكل دائم؟,delete_account,msa
أرغب في إيقاف حسابي نهائياً,delete_account,msa
ابي اقفل حسابي للأبد,delete_account,gulf
شيلوا حسابي ما عاد استخدمه,delete_account,gulf
كيف امسح الحساب من التطبيق؟,delete_account,gulf
هل تتوفر خدمة التوصيل للمنازل في القرى؟,delivery_options,msa
ما أنواع الشحن التي تقدمونها؟,delivery_options,msa
هل يمكن اختيار موعد محدد للتوصيل؟,delivery_options,msa
عندكم شحن سريع؟,delivery_options,gulf
توصلون لأبها ولا لا؟,delivery_options,gulf
اقدر اختار التوصيل بالليل؟,delivery_options,gulf
ما المدة المتوقعة لوصول الشحنة؟,delivery_period,msa
كم يوماً يستغرق التوصيل إلى الرياض؟,delivery_period,msa
متى سيصل الطلب إذا طلبته الآن؟,delivery_period,msa
كم ياخذ التوصيل لمكة؟,delivery_period,gulf
الشحن ياخذ كم يوم بالعادة؟,delivery_period,gulf
اذا طلبت الحين متى يجيني؟,delivery_period,gulf
أريد تغيير البريد الإلكتروني المسجل,edit_account,msa
كيف أعدّل العنوان في ملفي الشخصي؟,edit_account,msa
أرغب في تحديث كلمة المرور ورقم الهاتف,edit_account,msa
ابي احدث بياناتي في الحساب,edit_account,gulf
كيف اغير رقمي المسجل عندكم؟,edit_account,gulf
ودي اعدل تاريخ ميلادي في الحساب,edit_account,gulf
أرجو إرسال الفاتورة إلى بريدي الإلكتروني,get_invoice,msa
أحتاج فاتورة ضريبية لعملية الشراء,get_invoice,msa
من أين أستطيع تحميل الفاتورة؟,get_invoice,msa
ابي فاتورة الطلب عشان الشركة,get_invoice,gulf
وين القى الفاتورة حقت طلبي؟,get_invoice,gulf
طلعوا لي فاتورة لو سمحتوا,get_invoice,gulf
أطلب استرجاع قيمة المنتج المعيب,get_refund,msa
وصلني منتج مختلف وأريد استرداد أموالي,get_refund,msa
أرغب في إعادة المبلغ إلى بطاقتي,get_refund,msa
ابي استرجع فلوسي المنتج خربان,get_refund,gulf
رجعوا لي المبلغ المنتج مو نفس الصورة,get_refund,gulf
كيف اسوي طلب استرجاع فلوس؟,get_refund,gulf
أريد التسجيل في القائمة البريدية للعروض,newsletter_subscription,msa
كيف ألغي الاشتراك في الرسائل التسويقية؟,newsletter_subscription,msa
أرغب في استقبال آخر العروض على بريدي,newsletter_subscription,msa
وقفوا الايميلات اللي تجيني كل يوم,newsletter_subscription,gulf
ابي اشترك عشان يوصلني جديدكم,newsletter_subscription,gulf
كيف اطفي رسايل العروض؟,newsletter_subscription,gulf
تم رفض بطاقتي أثناء الدفع,payment_issue,msa
خُصم المبلغ ولم يتم تأكيد الطلب,payment_issue,msa
توجد مشكلة في إتمام عملية الدفع,payment_issue,msa
البطاقة ما تمشي عندكم ليش؟,payment_issue,gulf
دفعت والطلب ما انعمل والفلوس راحت,payment_issue,gulf
الدفع يطلع لي خطأ كل مرة,payment_issue,gulf
أرغب في شراء منتج من متجركم,place_order,msa
ما هي خطوات إتمام الطلب؟,place_order,msa
أريد طلب عدة منتجات دفعة واحدة,place_order,msa
ابي اطلب من عندكم ساعدوني,place_order,gulf
كيف اسوي طلب من التطبيق؟,place_order,gulf
ودي اشتري لابتوب وش اسوي؟,place_order,gulf
لا أتذكر كلمة السر الخاصة بحسابي,recover_password,msa
أحتاج إلى استعادة كلمة المرور,recover_password,msa
لا أستطيع تسجيل الدخول بسبب كلمة المرور,recover_password,msa
الرقم السري نسيته وش اسوي؟,recover_password,gulf
ابي اغير الباسورد لاني ناسيه,recover_password,gulf
ما اقدر ادخل حسابي نسيت الرمز,recover_password,gulf
تظهر لي مشكلة عند محاولة التسجيل,registration_problems,msa
لا يقبل الموقع بريدي الإلكتروني عند التسجيل,registration_problems,msa
فشلت عملية إنشاء الحساب أكثر من مرة,registration_problems,msa
ما يجيني كود التفعيل,registration_problems,gulf
التسجيل ما يزبط يقول الرقم مستخدم,registration_problems,gulf
كل ما اسجل يطلع لي خطأ,registration_problems,gulf
أرغب في تقييم تجربتي مع المتجر,review,msa
كيف أترك تعليقاً على المنتج؟,review,msa
أود مشاركة رأيي حول جودة المنتج,review,msa
ابي اعطي المنتج خمس نجوم وين اقيمه؟,review,gulf
ودي اكتب تعليق على الطلب,review,gulf
وين احط رأيي في المنتج؟,review,gulf
كيف أضيف عنواناً جديداً إلى حسابي؟,set_up_shipping_address,msa
أريد حفظ عنوان العمل في حسابي,set_up_shipping_address,msa
أرغب في تسجيل عنوان توصيل لأول مرة,set_up_shipping_address,msa
ابي احط عنواني في الحساب,set_up_shipping_address,gulf
كيف اسجل عنوان جديد عندكم؟,set_up_shipping_address,gulf
ودي اضيف عنوان بيتي الجديد في التطبيق,set_up_shipping_address,gulf
أريد تغيير نوع حسابي إلى حساب تجاري,switch_account,msa
هل يمكن الترقية إلى حساب مميز؟,switch_account,msa
كيف أنتقل من حساب فردي إلى حساب مؤسسة؟,switch_account,msa
ابي احول حسابي لحساب تاجر,switch_account,gulf
كيف ارجع حسابي عادي بدل الشركات؟,switch_account,gulf
ودي ابدل لحساب ثاني,switch_account,gulf
أريد معرفة مكان شحنتي الآن,track_order,msa
ما هي حالة طلبي الحالي؟,track_order,msa
أرجو تزويدي برقم تتبع الشحنة,track_order,msa
طلبي وين وصل؟,track_order,gulf
ابي اتتبع الشحنة حقتي,track_order,gulf
الطلب طلع ولا للحين؟,track_order,gulf
متى سيتم إيداع المبلغ المسترد في حسابي؟,track_refund,msa
أريد متابعة حالة استرجاع المبلغ,track_refund,msa
لم أستلم قيمة الاسترداد بعد,track_refund,msa
فلوسي متى ترجع؟,track_refund,gulf
الاسترجاع وين وصل للحين ما جاني شي,track_refund,gulf
طولتوا في رد المبلغ متى بيوصل؟,track_refund,gulf'''

synth = pd.read_csv(io.StringIO(SYNTH_CSV))
print(f"{len(synth)} synthetic tickets | {synth.intent.nunique()} intents | {synth.dialect.value_counts().to_dict()}")
synth.sample(8, random_state=0)

162 synthetic tickets | 27 intents | {'msa': 81, 'gulf': 81}


,text,intent,dialect
66,أريد إلغاء حسابي وحذف بياناتي,delete_account,msa
131,كل ما اسجل يطلع لي خطأ,registration_problems,gulf
145,هل يمكن الترقية إلى حساب مميز؟,switch_account,msa
7,هل يمكن استبدال المنتج في الطلب بلون آخر قبل ا...,change_order,msa
44,تأخر طلبي كثيراً ولم يتواصل معي أحد,complaint,msa
74,هل يمكن اختيار موعد محدد للتوصيل؟,delivery_options,msa
124,ابي اغير الباسورد لاني ناسيه,recover_password,gulf
159,فلوسي متى ترجع؟,track_refund,gulf


## 4 · Leakage check and cleaning
For every synthetic ticket we compute the **cosine similarity** to its closest Arabic dev/test ticket (e5 embeddings).
Anything above **0.95** is treated as a near-duplicate and **removed** before training — otherwise the model could score well by memorizing test-like sentences.

In [4]:
S = U.embed(synth.text, prefix="query: ")
T = U.embed(ar_all.text, prefix="query: ")
sim = S @ T.T                                    # synthetic x Arabic (vectors are normalized)
keep = sim.max(axis=1) <= 0.95

synth_clean = synth[keep].reset_index(drop=True)
S_clean = S[keep]
print(f"Near-duplicates removed: {(~keep).sum()} -> {len(synth_clean)} synthetic tickets kept\n")
pd.DataFrame({
    "removed_synthetic": synth.text[~keep].values,
    "closest_test_ticket": ar_all.text.values[sim[~keep].argmax(axis=1)],
    "cosine": sim[~keep].max(axis=1).round(3),
})

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Near-duplicates removed: 13 -> 149 synthetic tickets kept



,removed_synthetic,closest_test_ticket,cosine
0,أرغب في تعديل محتويات طلبي الحالي,أرغب في تعديل الكمية في طلبي الأخير,0.955
1,أرجو تحديث عنوان التسليم للطلب قبل خروجه,أريد تغيير عنوان التوصيل لطلبي,0.955
2,أريد إلغاء حسابي وحذف بياناتي,أرغب في حذف حسابي نهائياً,0.953
3,أرغب في إيقاف حسابي نهائياً,أرغب في حذف حسابي نهائياً,0.985
4,ابي احدث بياناتي في الحساب,ابي احول الحساب لحساب شركة,0.956
5,ودي اعدل تاريخ ميلادي في الحساب,ودي اغير اسمي في الحساب,0.965
6,لا أستطيع تسجيل الدخول بسبب كلمة المرور,نسيت كلمة المرور ولا أستطيع الدخول,0.967
7,تظهر لي مشكلة عند محاولة التسجيل,لا يصلني رمز التحقق عند التسجيل,0.951
8,لا يقبل الموقع بريدي الإلكتروني عند التسجيل,لا يصلني رمز التحقق عند التسجيل,0.952
9,أرغب في تسجيل عنوان توصيل لأول مرة,أرغب في حفظ عنوان افتراضي للتوصيل,0.951


## 5 · Train the classifiers
Every version uses the **same encoder** (`intfloat/multilingual-e5-small`, frozen) and the **same classifier** (logistic regression). Only the training data and τ change.

| Version | Training data | τ |
|---|---|---|
| **M0 · Baseline** | English | 0.5 (course default) |
| **M1 · Tuned τ** | English | chosen on validation |
| **M2 · + Arabic** | English + 162 synthetic Arabic | chosen on validation |
| **M3 · + Arabic (clean)** ⭐ | English + 149 synthetic Arabic (near-duplicates removed) | chosen on validation |

In [5]:
X_train = U.embed(train.text, prefix="query: ", cache_key="train")
X = {name: U.embed(df.text, prefix="query: ") for name, df in
     [("val", val), ("en_test", en_test), ("ar_dev", ar_dev), ("ar_test", ar_test)]}

def fit(Xs, ys):
    return LogisticRegression(max_iter=3000, C=20.0).fit(np.vstack(Xs), pd.concat(ys))

clf_en    = fit([X_train], [train.intent])
clf_mix   = fit([X_train, S], [train.intent, synth.intent])
clf_clean = fit([X_train, S_clean], [train.intent, synth_clean.intent])
print("✅ Trained clf_en, clf_mix, clf_clean")

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

✅ Trained clf_en, clf_mix, clf_clean


## 6 · Choose τ on validation data
τ is the confidence threshold: below it the ticket goes to a human (`(abstain)`).
We pick the τ with the best **mean macro-F1 of English val and Arabic dev** — never the test set.

In [6]:
def predict(clf, Xm, tau):
    P = clf.predict_proba(Xm)
    conf = P.max(axis=1)
    return np.where(conf >= tau, clf.classes_[P.argmax(axis=1)], "(abstain)"), conf

def mf1(y, p):
    return f1_score(y, p, average="macro", zero_division=0)

def choose_tau(clf):
    rows = []
    for tau in np.round(np.arange(0.05, 0.51, 0.05), 2):
        f_en = mf1(val.intent, predict(clf, X["val"], tau)[0])
        f_ar = mf1(ar_dev.intent, predict(clf, X["ar_dev"], tau)[0])
        rows.append({"tau": tau, "val_en_f1": round(f_en, 3), "dev_ar_f1": round(f_ar, 3), "mean": round((f_en + f_ar) / 2, 3)})
    sweep = pd.DataFrame(rows)
    return float(sweep.loc[sweep["mean"].idxmax(), "tau"]), sweep

tau_en, sweep_en = choose_tau(clf_en)
tau_mix, _ = choose_tau(clf_mix)
tau_clean, _ = choose_tau(clf_clean)
print(f"τ  M1 = {tau_en} | M2 = {tau_mix} | M3 = {tau_clean}")
sweep_en

τ  M1 = 0.25 | M2 = 0.05 | M3 = 0.05


,tau,val_en_f1,dev_ar_f1,mean
0,0.05,0.997,0.427,0.712
1,0.10,0.997,0.415,0.706
2,0.15,0.997,0.420,0.709
3,0.20,0.997,0.440,0.719
4,0.25,0.997,0.452,0.725
5,0.30,0.997,0.451,0.724
6,0.35,0.961,0.367,0.664
7,0.40,0.960,0.325,0.643
8,0.45,0.960,0.231,0.596
9,0.50,0.959,0.195,0.577


## 7 · Final test: all versions on the same test set
This is the first time the test sets are used. Bootstrap 95% CIs (500 resamples).

In [7]:
VERSIONS = {
    "M0_baseline":     (clf_en,    0.5),
    "M1_tuned_tau":    (clf_en,    tau_en),
    "M2_arabic_data":  (clf_mix,   tau_mix),
    "M3_arabic_clean": (clf_clean, tau_clean),
}

parts = []
for name, (clf, tau) in VERSIONS.items():
    for split, df, lang in [("en_test", en_test, "en"), ("ar_test", ar_test, "ar")]:
        pred, conf = predict(clf, X[split], tau)
        parts.append(pd.DataFrame({"model": name, "text": df.text, "y_true": df.intent, "y_pred": pred,
                                   "confidence": conf.round(3), "lang": lang,
                                   "dialect": df.dialect.values if lang == "ar" else "en"}))
preds = pd.concat(parts, ignore_index=True)
for name in VERSIONS:
    preds[preds.model == name].drop(columns="model").to_csv(OUT / f"preds_{name}.csv", index=False)

def summarize(g):
    f, lo, hi = U.bootstrap_ci(g.y_true, g.y_pred, mf1, n=500)
    return {"n": len(g), "macro_f1": round(f, 3), "95% CI": f"[{lo:.3f}, {hi:.3f}]",
            "abstain %": f"{(g.y_pred == '(abstain)').mean():.0%}"}

rows = []
for name in VERSIONS:
    p = preds[preds.model == name]
    for label, g in [("EN", p[p.lang == "en"]), ("AR", p[p.lang == "ar"]),
                     ("AR · MSA", p[p.dialect == "msa"]), ("AR · Gulf", p[p.dialect == "gulf"])]:
        rows.append({"model": name, "slice": label, **summarize(g)})
results = pd.DataFrame(rows)
results.to_csv(OUT / "results_comparison.csv", index=False)
results.pivot(index="model", columns="slice", values="macro_f1")[["EN", "AR", "AR · MSA", "AR · Gulf"]]

slice,EN,AR,AR · MSA,AR · Gulf
model,,,,
M0_baseline,0.958,0.262,0.357,0.036
M1_tuned_tau,0.996,0.398,0.423,0.274
M2_arabic_data,0.996,0.711,0.753,0.568
M3_arabic_clean,0.996,0.733,0.753,0.648


In [8]:
results

,model,slice,n,macro_f1,95% CI,abstain %
0,M0_baseline,EN,1080,0.958,"[0.954, 0.962]",1%
1,M0_baseline,AR,54,0.262,"[0.122, 0.325]",78%
2,M0_baseline,AR · MSA,27,0.357,"[0.222, 0.474]",59%
3,M0_baseline,AR · Gulf,27,0.036,"[0.000, 0.062]",96%
4,M1_tuned_tau,EN,1080,0.996,"[0.993, 0.999]",0%
5,M1_tuned_tau,AR,54,0.398,"[0.243, 0.470]",37%
6,M1_tuned_tau,AR · MSA,27,0.423,"[0.265, 0.550]",26%
7,M1_tuned_tau,AR · Gulf,27,0.274,"[0.139, 0.402]",48%
8,M2_arabic_data,EN,1080,0.996,"[0.993, 0.999]",0%
9,M2_arabic_data,AR,54,0.711,"[0.523, 0.790]",0%


In [9]:
!python capstone/evaluate.py results/preds_M3_arabic_clean.csv --slice lang --slice dialect 2>&1 | head -12

       slice    n  accuracy  macro_f1  macro_f1_95ci
         all 1134     0.985     0.985 [0.977, 0.991]
     lang=ar   54     0.759     0.733 [0.576, 0.802]
     lang=en 1080     0.996     0.996 [0.993, 0.999]
  dialect=en 1080     0.996     0.996 [0.993, 0.999]
dialect=gulf   27     0.704     0.648 [0.467, 0.776]
 dialect=msa   27     0.815     0.753 [0.578, 0.911]

Most frequent confusions:
                   true                    pred  count
    check_refund_policy              get_refund      2
           change_order             track_order      1


## 8 · Model selection: which text encoder?
Same training data as M3, same classifier, same test sets — **only the encoder changes**. No τ (pure argmax), so we compare the encoders alone. Latency = encode + classify one Arabic ticket on CPU.

| Encoder | Why it is in the comparison |
|---|---|
| TF-IDF (char 2–4 grams) | No embeddings at all — do embeddings matter? |
| paraphrase-multilingual-MiniLM | A competing multilingual encoder of similar size |
| **multilingual-e5-small** | Our encoder (course default) |
| multilingual-e5-base | Same family, ~2× bigger — does size pay off? |

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tr_text = pd.concat([train.text, synth_clean.text]).tolist()
tr_y    = pd.concat([train.intent, synth_clean.intent]).tolist()

def embedder(model, prefix):
    return lambda texts: U.embed(list(texts), prefix=prefix, model=model)

tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2).fit(tr_text)
ENCODERS = {
    "TF-IDF (no embeddings)": lambda t: tfidf.transform(list(t)),
    "MiniLM-multilingual":    embedder("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", ""),
    "e5-small (ours)":        embedder("intfloat/multilingual-e5-small", "query: "),
    "e5-base (2x bigger)":    embedder("intfloat/multilingual-e5-base", "query: "),
}

rows = []
for name, enc in ENCODERS.items():
    print("⏳", name)
    clf = LogisticRegression(max_iter=3000, C=20.0).fit(enc(tr_text), tr_y)
    row = {"encoder": name}
    for label, df in [("EN", en_test), ("AR", ar_test)]:
        f, lo, hi = U.bootstrap_ci(df.intent, clf.predict(enc(df.text)), mf1, n=500)
        row[f"{label} macro_f1"] = round(f, 3)
        if label == "AR":
            row["AR 95% CI"] = f"[{lo:.3f}, {hi:.3f}]"
    sample = ar_test.text.tolist()[:30]
    t0 = time.perf_counter()
    for s in sample:
        clf.predict(enc([s]))
    row["ms / ticket"] = round((time.perf_counter() - t0) / len(sample) * 1000, 1)
    rows.append(row)

comparison = pd.DataFrame(rows)
comparison.to_csv(OUT / "model_comparison.csv", index=False)
comparison

⏳ TF-IDF (no embeddings)
⏳ MiniLM-multilingual


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/87 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

,encoder,EN macro_f1,AR macro_f1,AR 95% CI,ms / ticket
0,TF-IDF (no embeddings),0.996,0.662,"[0.482, 0.742]",1.0
1,MiniLM-multilingual,0.994,0.752,"[0.588, 0.829]",30.3
2,e5-small (ours),0.996,0.733,"[0.576, 0.802]",31.3
3,e5-base (2x bigger),0.997,0.760,"[0.586, 0.836]",112.9


**Reading the table**
- Embeddings beat TF-IDF on Arabic, at a latency cost.
- The three embedding encoders overlap within their confidence intervals on 54 Arabic tickets — the differences are not significant.
- e5-base is the most accurate but ~3.6× slower than e5-small (112.9 vs 31.3 ms).
- We keep **e5-small**: chosen *before* looking at the test set, comparable accuracy, small and CPU-friendly. Switching to whichever encoder wins on the test set would be selecting on test data.

## 9 · Error analysis (M3)
Failures are categorized with the cause list from Lab D3_4:
**ambiguous** (two intents a human could confuse) · **coverage** (wording/dialect missing from training) · **model error** (clear ticket, wrong answer) · **label noise** (a training label is wrong) · **out of scope**.

In [14]:
CAUSES = {
    "انتقلت إلى شقة جديدة، كيف أحدث عنوان الشحن للطلب الحالي؟": ("ambiguous", "change vs set-up shipping address are near-synonyms"),
    "لو رفضت الاستلام بيخصمون علي شي؟": ("coverage", "indirect phrasing: 'refuse delivery' instead of 'cancel'"),
    "اقدر ادفع بأبل باي؟": ("coverage", "brand name 'Apple Pay' never seen in training"),
    "هل يمكن استرداد المبلغ للمنتجات المخفضة؟": ("ambiguous", "policy question that contains the word 'refund'"),
    "وش الشروط عشان ارجع القطعة واسترد فلوسي؟": ("model error", "'conditions' is explicit, but 'get my money back' dominated"),
    "ما هي خطوات التسجيل في الموقع؟": ("ambiguous", "'registration' appears in both create_account and registration_problems"),
    "اقدر استلم الطلب من نقطة استلام؟": ("coverage", "concept 'pickup point' never seen in training"),
    "رجعوا لي فلوسي بسرعة لو سمحت": ("ambiguous", "urgency reads like refund tracking"),
    "انسحبت الفلوس مرتين على نفس الطلب عاجل": ("coverage", "Gulf verb 'انسحبت' (withdrawn) not in training"),
    "كيف أعيد تعيين كلمة المرور؟": ("label noise", "a synthetic edit_account example mentions updating the password"),
    "حسابي انقفل وما اقدر ادخل": ("coverage", "'locked' is close to 'close my account' in delete_account examples"),
    "ما هي حالة طلب الاسترداد الخاص بي؟": ("model error", "clear refund-status request"),
        "صار لي عشرة ايام وما رجع المبلغ": ("coverage", "implicit complaint, no explicit question"),
    "edtiing order A-95256": ("coverage", "typo 'edtiing' not covered by training"),
    "i exepct a compensation of SAR1250": ("label noise", "asks for compensation: get_refund fits better than the gold label"),
    "i cannot find bill #85632 can usendit to me": ("ambiguous", "get_invoice vs check_invoice (the Lab D3_4 example)"),
    "am I entitled to a reimbursement?": ("ambiguous", "eligibility question that contains 'reimbursement'"),
}

p = preds[preds.model == "M3_arabic_clean"]
for lg in ["ar", "en"]:
    q = p[p.lang == lg]
    print(f"M3 errors {lg.upper()}: {(q.y_true != q.y_pred).sum()} / {len(q)}")

err = p[p.y_true != p.y_pred]
errors = pd.concat([err[err.lang == "ar"], err[err.lang == "en"]])[["text", "lang", "dialect", "y_true", "y_pred", "confidence"]].reset_index(drop=True)
errors["cause"] = errors.text.map(lambda t: CAUSES.get(t, ("to review", ""))[0])
errors["note"] = errors.text.map(lambda t: CAUSES.get(t, ("", ""))[1])
errors.to_csv(OUT / "error_analysis_M3.csv", index=False)
print("\n", errors.cause.value_counts().to_string())
print(f"\nMax confidence among errors: {errors.confidence.max():.2f}")
errors

M3 errors AR: 13 / 54
M3 errors EN: 4 / 1080

 cause
coverage       7
ambiguous      6
model error    2
label noise    2

Max confidence among errors: 0.78


,text,lang,dialect,y_true,y_pred,confidence,cause,note
0,انتقلت إلى شقة جديدة، كيف أحدث عنوان الشحن للط...,ar,msa,change_shipping_address,set_up_shipping_address,0.513,ambiguous,change vs set-up shipping address are near-syn...
1,لو رفضت الاستلام بيخصمون علي شي؟,ar,gulf,check_cancellation_fee,cancel_order,0.190,coverage,indirect phrasing: 'refuse delivery' instead o...
2,اقدر ادفع بأبل باي؟,ar,gulf,check_payment_methods,get_invoice,0.358,coverage,brand name 'Apple Pay' never seen in training
3,هل يمكن استرداد المبلغ للمنتجات المخفضة؟,ar,msa,check_refund_policy,get_refund,0.388,ambiguous,policy question that contains the word 'refund'
4,وش الشروط عشان ارجع القطعة واسترد فلوسي؟,ar,gulf,check_refund_policy,track_refund,0.211,model error,"'conditions' is explicit, but 'get my money ba..."
5,ما هي خطوات التسجيل في الموقع؟,ar,msa,create_account,registration_problems,0.401,ambiguous,'registration' appears in both create_account ...
6,اقدر استلم الطلب من نقطة استلام؟,ar,gulf,delivery_options,track_order,0.154,coverage,concept 'pickup point' never seen in training
7,رجعوا لي فلوسي بسرعة لو سمحت,ar,gulf,get_refund,track_refund,0.141,ambiguous,urgency reads like refund tracking
8,انسحبت الفلوس مرتين على نفس الطلب عاجل,ar,gulf,payment_issue,cancel_order,0.198,coverage,Gulf verb 'انسحبت' (withdrawn) not in training
9,كيف أعيد تعيين كلمة المرور؟,ar,msa,recover_password,edit_account,0.483,label noise,a synthetic edit_account example mentions upda...


**Takeaways**
- Remaining errors are mostly Arabic (13 of 54 Arabic vs 4 of 1,080 English); most are **coverage** (Gulf words, brands, typos, concepts missing from training) or **ambiguous** intent pairs.
- All Arabic errors have low confidence (≤ 0.51), so a higher τ would route them to a human instead of mislabeling them (automation vs safety trade-off). Two English errors are **confident mistakes** (0.73, 0.78) that no threshold would catch.
- Next step: real Arabic/Gulf tickets rather than synthetic ones.

## 10 · Save results and the final model
The final model (M3 classifier + τ + label list) is saved for the Gradio demo. Everything is copied to Google Drive → `NLP_capstone_final/`.

In [13]:
import joblib, shutil
joblib.dump({"clf": clf_clean, "tau": tau_clean, "encoder": "intfloat/multilingual-e5-small",
             "prefix": "query: "}, OUT / "intent_clf_M3.joblib")
synth_clean.to_csv(OUT / "synth_ar_train_clean.csv", index=False)
print(sorted(f.name for f in OUT.iterdir()))

from google.colab import drive
drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/NLP_capstone_final")
dest.mkdir(exist_ok=True)
for f in OUT.iterdir():
    shutil.copy(f, dest / f.name)
print(f"✅ Saved {len(list(OUT.iterdir()))} files to Drive → NLP_capstone_final/")

['error_analysis_M3.csv', 'intent_clf_M3.joblib', 'model_comparison.csv', 'preds_M0_baseline.csv', 'preds_M1_tuned_tau.csv', 'preds_M2_arabic_data.csv', 'preds_M3_arabic_clean.csv', 'results_comparison.csv', 'synth_ar_train_clean.csv']
Mounted at /content/drive
✅ Saved 9 files to Drive → NLP_capstone_final/
